Environment Setup

In [ ]:
import os
import pandas as pd
import numpy as np
import cv2
import glob
import shutil
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import Conv2D, Flatten, Dense, Dropout
from tensorflow.keras.callbacks import ModelCheckpoint
from tensorflow.keras.losses import MeanSquaredError
from sklearn.model_selection import train_test_split
from google.colab import drive
IMG_HEIGHT, IMG_WIDTH = 96, 160
# --- 1. MOUNT & CONFIGURATION ---
try:
    drive.mount('/content/drive')
except:
    # If already mounted or if you face the ValueError we discussed
    !rm -rf /content/drive
    drive.mount('/content/drive', force_remount=True)

# Define Permanent Master Paths
MASTER_BASE_DIR = "/content/drive/MyDrive/KartProject/master_dataset"
INCOMING_DIR = "/content/drive/MyDrive/KartProject/incoming_data"
MODEL_SAVE_PATH = "/content/drive/MyDrive/KartProject/best_kart_model.keras"

# Mapping for compatibility with existing modules
EXTRACT_DIR = MASTER_BASE_DIR
MASTER_IMAGE_DIR = os.path.join(EXTRACT_DIR, "images")
MASTER_CSV_PATH = os.path.join(EXTRACT_DIR, "driving_log.csv")
MASTER_MASK_DIR = os.path.join(MASTER_BASE_DIR, "masks") # New home for extracted masks
# Ensure folders exist
for folder in [MASTER_IMAGE_DIR, INCOMING_DIR, MASTER_MASK_DIR]:
    os.makedirs(folder, exist_ok=True)

# --- SETTING: TOGGLE TRAINING MODE ---
# Set to True to train ONLY on the ZIPs you just uploaded.
# Set to False to train on everything you've ever collected.
USE_ONLY_NEW_DATA = True

# --- 2. DATA INGESTION FUNCTION (MODIFIED) ---
def merge_hybrid_team_data():
    all_zips = glob.glob(os.path.join(INCOMING_DIR, "*.zip"))
    new_session_data = pd.DataFrame(columns=['image', 'mask', 'steering'])

    # Load or Create Master CSV
    if os.path.exists(MASTER_CSV_PATH):
        master_df = pd.read_csv(MASTER_CSV_PATH)
    else:
        master_df = pd.DataFrame(columns=['image', 'mask', 'steering'])

    for zip_file in all_zips:
        zip_name = os.path.basename(zip_file).replace(".zip", "")
        print(f"📦 Processing: {zip_name}")

        temp_path = "/content/temp_unzip"
        if os.path.exists(temp_path): shutil.rmtree(temp_path)
        !unzip -q "{zip_file}" -d {temp_path}

        # --- FIX: Specifically target the 'data' folder inside the unzip path ---
        internal_data_dir = os.path.join(temp_path, "data")

        # Check if the 'data' folder exists, otherwise fallback to root
        base_search_path = internal_data_dir if os.path.exists(internal_data_dir) else temp_path

        csv_file_path = os.path.join(base_search_path, "driving_log.csv")
        npz_file_path = os.path.join(base_search_path, "masks.npz")

        if not os.path.exists(csv_file_path) or not os.path.exists(npz_file_path):
            print(f"⚠️ Could not find CSV or NPZ in {base_search_path}. Skipping.")
            continue

        temp_csv = pd.read_csv(csv_file_path)
        with np.load(npz_file_path) as data:
            all_masks = data['masks']

        # 2. Process each frame
        for idx, row in temp_csv.iterrows():
            clean_name = os.path.basename(row['image']).replace(".jpg", "")
            unique_prefix = f"{zip_name}_{clean_name}"

            # Source image is in internal_data_dir/images/
            old_img_path = os.path.join(base_search_path, 'images', os.path.basename(row['image']))
            new_img_name = f"{unique_prefix}.jpg"
            new_mask_name = f"{unique_prefix}_mask.png"

            # Move RGB Image
            if os.path.exists(old_img_path):
                shutil.move(old_img_path, os.path.join(MASTER_IMAGE_DIR, new_img_name))

            # Save Mask from NPZ
            cv2.imwrite(os.path.join(MASTER_MASK_DIR, new_mask_name), all_masks[idx])

            # Update the dataframe
            temp_csv.at[idx, 'image'] = new_img_name
            temp_csv.at[idx, 'mask'] = new_mask_name

        # Concatenate batch
        batch_df = temp_csv[['image', 'mask', 'steering']]
        new_session_data = pd.concat([new_session_data, batch_df], ignore_index=True)
        master_df = pd.concat([master_df, batch_df], ignore_index=True)

        # Archive ZIP
        os.makedirs(os.path.join(INCOMING_DIR, "archived"), exist_ok=True)
        shutil.move(zip_file, os.path.join(INCOMING_DIR, "archived", os.path.basename(zip_file)))

    master_df.to_csv(MASTER_CSV_PATH, index=False)
    print(f"Master library updated! Total frames: {len(master_df)}")

    return new_session_data if USE_ONLY_NEW_DATA else master_df

# --- 3. EXECUTION ---
# Catch the dataframe returned by the function
current_training_df = merge_hybrid_team_data()

Mounted at /content/drive
Master library updated! Total frames: 10207


Data Loading and Preprocessing

Model Architecture

In [ ]:
#@title Model Options
CREATE_NEW_MODEL = True #@param {type:"boolean"}
OVERRIDE_SAVED_MODEL = True #@param {type:"boolean"}
MODEL_SAVE_PATH = "best_kart_model.keras"
from tensorflow.keras.layers import Input, Conv2D, Flatten, Dense, Dropout, UpSampling2D, \
    concatenate, SpatialDropout2D, Multiply, Add, Resizing
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.models import Model

def build_pro_kart_model():
    # INPUT: 3 stacked frames (96, 160, 9 channels)
    inputs = Input(shape=(96, 160, 9), name="stacked_frames")

    # --- ENCODER (The Shared Backbone) ---
    c1 = Conv2D(24, (5, 5), strides=(2, 2), padding='same', activation='relu')(inputs)
    c2 = Conv2D(36, (5, 5), strides=(2, 2), padding='same', activation='relu')(c1)
    c3 = Conv2D(48, (5, 5), strides=(2, 2), padding='same', activation='relu')(c2)
    shared = Conv2D(64, (3, 3), padding='same', activation='elu')(c3) # 12x20

    # --- DECODER (With Spatial Dropout) ---
    d1 = UpSampling2D((2, 2), interpolation='nearest')(shared)
    d1 = SpatialDropout2D(0.1)(d1) # Prevents "checkerboard" artifacts
    d1 = Conv2D(48, (3, 3), activation='relu', padding='same')(d1)

    d2 = UpSampling2D((2, 2), interpolation='nearest')(d1)
    d2 = Conv2D(24, (3, 3), activation='relu', padding='same')(d2)

    d3 = UpSampling2D((2, 2), interpolation='nearest')(d2)
    # The mask head output (Sigmoid for binary road/not-road)
    mask_output = Conv2D(1, (1, 1), activation='sigmoid', name="mask_output")(d3)

    # --- SOFT ATTENTION GATE (1 + Mask) ---
    # Downscale the mask to match the shared features (12x20)
    attention_map = Resizing(12, 20)(mask_output)

    # Soft Attention: (Features * Mask) + Features
    # This keeps 'raw' features alive so it can still see cars/signs
    gated_features = Multiply()([shared, attention_map])
    soft_attention_features = Add()([gated_features, shared])

    # --- STEERING HEAD (Temporal + Spatial) ---
    s = Flatten()(soft_attention_features)
    s = Dense(100, activation='relu')(s)
    s = Dropout(0.5)(s)
    steering_output = Dense(1, name="steering_output")(s)

    model = Model(inputs=inputs, outputs=[steering_output, mask_output])
    model.compile(optimizer=Adam(learning_rate=1e-4),
              loss={'steering_output': 'mse', 'mask_output': 'binary_crossentropy'},
              loss_weights={'steering_output': 1.0, 'mask_output': 50.0})
    return model
if CREATE_NEW_MODEL:
    print("🚀 Creating a fresh 9-channel model...")
    model = build_pro_kart_model()
    # Force a save immediately to overwrite the old 3-channel file
    if OVERRIDE_SAVED_MODEL:
        model.save(MODEL_SAVE_PATH)
        print(f"✅ Overwrote {MODEL_SAVE_PATH} with fresh 9-channel architecture.")
else:
    try:
        model = tf.keras.models.load_model(MODEL_SAVE_PATH)
        print("📁 Loaded existing model from disk.")
    except Exception as e:
        print(f"❌ Failed to load model: {e}. Building new instead.")
        model = build_pro_kart_model()

model.summary()

🚀 Creating a fresh 9-channel model...
✅ Overwrote best_kart_model.keras with fresh 9-channel architecture.


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ stacked_frames      │ (None, 96, 160,   │          0 │ -                 │
│ (InputLayer)        │ 9)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d (Conv2D)     │ (None, 48, 80,    │      5,424 │ stacked_frames[0… │
│                     │ 24)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_1 (Conv2D)   │ (None, 24, 40,    │     21,636 │ conv2d[0][0]      │
│                     │ 36)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_2 (Conv2D)   │ (None, 12, 20,    │     43,248 │ conv2d_1[0][0]    │
│                     │ 48)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_3 (Conv2D)   │ (None, 12, 20,    │     27,712 │ conv2d_2[0][0]    │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ up_sampling2d       │ (None, 24, 40,    │          0 │ conv2d_3[0][0]    │
│ (UpSampling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ spatial_dropout2d   │ (None, 24, 40,    │          0 │ up_sampling2d[0]… │
│ (SpatialDropout2D)  │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_4 (Conv2D)   │ (None, 24, 40,    │     27,696 │ spatial_dropout2… │
│                     │ 48)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ up_sampling2d_1     │ (None, 48, 80,    │          0 │ conv2d_4[0][0]    │
│ (UpSampling2D)      │ 48)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_5 (Conv2D)   │ (None, 48, 80,    │     10,392 │ up_sampling2d_1[… │
│                     │ 24)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ up_sampling2d_2     │ (None, 96, 160,   │          0 │ conv2d_5[0][0]    │
│ (UpSampling2D)      │ 24)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ mask_output         │ (None, 96, 160,   │         25 │ up_sampling2d_2[… │
│ (Conv2D)            │ 1)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ resizing (Resizing) │ (None, 12, 20, 1) │          0 │ mask_output[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multiply (Multiply) │ (None, 12, 20,    │          0 │ conv2d_3[0][0],   │
│                     │ 64)               │            │ resizing[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, 12, 20,    │          0 │ multiply[0][0],   │
│                     │ 64)               │            │ conv2d_3[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten (Flatten)   │ (None, 15360)     │          0 │ add[0][0]         │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 100)       │  1,536,100 │ flatten[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 100)       │          0 │ dense[0][0]     

 Total params: 1,672,334 (6.38 MB)

 Trainable params: 1,672,334 (6.38 MB)

 Non-trainable params: 0 (0.00 B)

Generate Digestible Data for Model

In [ ]:
import numpy as np
import os
import pandas as pd
import tensorflow as tf
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
# --- 1. THE BALANCER ---
def balance_steering_data(df, num_bins=25, samples_per_bin=600):
    hist, bins = np.histogram(df['steering'], num_bins)
    keep_list = []
    for i in range(num_bins):
        bin_data = df[(df['steering'] >= bins[i]) & (df['steering'] <= bins[i+1])]
        if len(bin_data) > samples_per_bin:
            bin_data = bin_data.sample(samples_per_bin)
        keep_list.append(bin_data)
    return pd.concat(keep_list).reset_index(drop=True)

# --- 2. THE AUGMENTER (9-Channel Aware) ---
def augment_data(stacked_img, mask, steering):
    # --- 1. LIGHTING (Pixel-wise) ---
    # Helps the AI handle sunlight, shadows, and camera exposure changes
    stacked_img = tf.image.random_brightness(stacked_img, max_delta=0.2)
    stacked_img = tf.image.random_contrast(stacked_img, lower=0.7, upper=1.3)

    # --- 2. VIBRATION / ROW-SHIFT (Geometric Noise) ---
    # Simulates rolling shutter "jello" by shifting rows horizontally.
    # This is much more realistic for a 25mph kart than standard blur.
    if tf.random.uniform(()) > 0.5:
        h, w = 96, 160
        max_shift = 4 # Shift rows by up to 4 pixels left or right

        # Generate random shifts for every row
        shifts = tf.random.uniform([h], minval=-max_shift, maxval=max_shift, dtype=tf.int32)

        # Apply shift row-by-row using tf.roll
        # This keeps the image data but "jiggles" the lines
        stacked_img = tf.map_fn(
            lambda i: tf.roll(stacked_img[i], shift=shifts[i], axis=0),
            tf.range(h),
            fn_output_signature=tf.float32
        )

    # --- 3. OCCLUSION (16x16 Black Box) ---
    # Simulates a piece of mud, a leaf, or a camera glitch.
    # Forces the AI to look at the rest of the road to make decisions.
    if tf.random.uniform(()) > 0.5:
        h, w = 96, 160
        box_size = 16

        # Pick random top-left corner
        top = tf.random.uniform([], 0, h - box_size, dtype=tf.int32)
        left = tf.random.uniform([], 0, w - box_size, dtype=tf.int32)

        # Create coordinates for the box
        # We create a mask the same size as the image
        row_indices = tf.range(h)
        col_indices = tf.range(w)

        row_mask = tf.logical_and(row_indices >= top, row_indices < top + box_size)
        col_mask = tf.logical_and(col_indices >= left, col_indices < left + box_size)

        # Combine into a 2D boolean mask
        box_mask = tf.logical_and(row_mask[:, tf.newaxis], col_mask[tf.newaxis, :])

        # Apply only to the stacked_img (the 'Question'), not the mask (the 'Answer')
        stacked_img = tf.where(box_mask[..., tf.newaxis], tf.zeros_like(stacked_img), stacked_img)

    # --- 4. SPATIAL FLIP (Must be applied to both Image and Mask) ---
    if tf.random.uniform(()) > 0.5:
        stacked_img = tf.image.flip_left_right(stacked_img)
        mask = tf.image.flip_left_right(mask)
        steering = -steering

    return stacked_img, mask, steering

# --- 3. THE STACKING LOADER ---
def load_and_stack(img_paths, mask_path, steering):
    # --- 1. Load Images (t, t-1, t-2) using direct indexing ---
    def _parse_img(path):
        img = tf.io.read_file(path)
        img = tf.image.decode_jpeg(img, channels=3)
        img = tf.image.convert_image_dtype(img, tf.float32)
        return tf.image.resize(img, [IMG_HEIGHT, IMG_WIDTH])

    # Explicitly pull paths to avoid the 'Iterating over a symbolic Tensor' error
    img_t   = _parse_img(img_paths[0])
    img_t_1 = _parse_img(img_paths[1])
    img_t_2 = _parse_img(img_paths[2])

    stacked_img = tf.concat([img_t, img_t_1, img_t_2], axis=-1)

    # --- 2. Load Mask and Target Semantic IDs ---
    mask_raw = tf.io.read_file(mask_path)
    mask_raw = tf.image.decode_png(mask_raw, channels=1)
    mask_raw = tf.cast(mask_raw, tf.int32) # Keep as IDs

    # Use nearest neighbor to keep the IDs 128, 255, etc. pure
    mask_raw = tf.image.resize(mask_raw, [IMG_HEIGHT, IMG_WIDTH], method='nearest')

    # --- 3. Semantic Mapping using your IDs ---
    # Road: 128, Lane Line: 255, Crosswalk: 189
    is_road = tf.equal(mask_raw, 128)
    is_lane = tf.equal(mask_raw, 255)
    is_cross = tf.equal(mask_raw, 189)

    drivable = tf.logical_or(is_road, tf.logical_or(is_lane, is_cross))
    mask = tf.where(drivable, 1.0, 0.0)

    return stacked_img, mask, steering

# --- 4. DATASET PREPARATION ---
def prepare_stacked_dataset(df, batch_size=32, augment=False):
    img_sequences = []
    mask_paths = []
    steerings = []

    # Shift indices to create the 3-frame "window"
    # We use .iloc values to ensure we get the actual filenames
    for i in range(2, len(df)):
        seq = [
            os.path.join(MASTER_IMAGE_DIR, df.iloc[i]['image']),   # t
            os.path.join(MASTER_IMAGE_DIR, df.iloc[i-1]['image']), # t-1
            os.path.join(MASTER_IMAGE_DIR, df.iloc[i-2]['image'])  # t-2
        ]
        img_sequences.append(seq)
        mask_paths.append(os.path.join(MASTER_MASK_DIR, df.iloc[i]['mask']))
        steerings.append(df.iloc[i]['steering'])

    ds = tf.data.Dataset.from_tensor_slices((img_sequences, mask_paths, steerings))

    if augment:
        ds = ds.shuffle(buffer_size=min(len(img_sequences), 1000))

    ds = ds.map(load_and_stack, num_parallel_calls=tf.data.AUTOTUNE)

    if augment:
        ds = ds.map(augment_data, num_parallel_calls=tf.data.AUTOTUNE)

    # Structure for the Dual-Head Model
    ds = ds.map(lambda i, m, s: (i, {"steering_output": s, "mask_output": m}))

    return ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)

In [ ]:
def verify_stacked_data(dataset):
    # Pull one batch
    for images, targets in dataset.take(1):
        img_batch = images.numpy()
        mask_batch = targets['mask_output'].numpy()
        steer_batch = targets['steering_output'].numpy()

        # Let's look at the first sample in the batch
        stack = img_batch[0]
        mask = mask_batch[0]
        steer = steer_batch[0]

        fig, axes = plt.subplots(1, 4, figsize=(20, 5))

        # Slicing the 9 channels back into 3 RGB images
        axes[0].imshow(stack[:, :, 0:3])
        axes[0].set_title(f"Current (t)\nSteer: {steer:.2f}")

        axes[1].imshow(stack[:, :, 3:6])
        axes[1].set_title("Previous (t-1)")

        axes[2].imshow(stack[:, :, 6:9])
        axes[2].set_title("Older (t-2)")

        axes[3].imshow(mask.squeeze(), cmap='gray')
        axes[3].set_title("Current Mask")

        for ax in axes:
            ax.axis('off')

        plt.show()
        print(f"Tensor Shape: {stack.shape}")
        print(f"Steering Value: {steer}")

# 1. Balance
balanced_df = balance_steering_data(current_training_df)

# Add a check for empty DataFrame
if balanced_df.empty:
    print("⚠️ Warning: No data available for training after balancing. Please ensure 'incoming_data' has new ZIP files or set USE_ONLY_NEW_DATA to False in the setup cell (cell `wvTFjRWxeIrN`).")
    # Create empty datasets to prevent further errors, though training will be effectively skipped.
    train_ds = tf.data.Dataset.from_tensor_slices((np.array([]).reshape(0,96,160,9), {'steering_output': [], 'mask_output': []})).batch(1)
    val_ds = tf.data.Dataset.from_tensor_slices((np.array([]).reshape(0,96,160,9), {'steering_output': [], 'mask_output': []})).batch(1)
else:
    # 2. Split (Using shuffle=False because order matters for temporal stacking!)
    # We split by 'index' to keep sequences together within the sets
    train_df, val_df = train_test_split(balanced_df, test_size=0.2, shuffle=False)

    # 3. Build Datasets
    train_ds = prepare_stacked_dataset(train_df, batch_size=32, augment=True)
    val_ds = prepare_stacked_dataset(val_df, batch_size=32, augment=False)

print("💎 Datasets are in memory and ready." if not balanced_df.empty else "No datasets created due to empty input data.")

ValueError: With n_samples=0, test_size=0.2 and train_size=None, the resulting train set will be empty. Adjust any of the aforementioned parameters.

Model Training & Checkpointing

In [ ]:
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau, TensorBoard
import datetime

# --- 1. SETTINGS ---
EPOCHS = 20
BATCH_SIZE = 32
MODEL_NAME = "pro_kart_v1_stacked"

# --- 2. CALLBACKS (The Safety Net) ---
# Save the best model based on validation steering performance
checkpoint = ModelCheckpoint(
    f'{MODEL_NAME}.keras',
    monitor='val_steering_output_loss',
    save_best_only=True,
    mode='min',
    verbose=1
)

# Stop training if the model stops improving (prevents overfitting)
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True,
    verbose=1
)

# Learning Rate Decay: If the loss plateaus, make the steps smaller
reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.2,
    patience=3,
    min_lr=1e-6,
    verbose=1
)

# (Optional) For visualization in TensorBoard
log_dir = "logs/fit/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensorboard_callback = tf.keras.callbacks.TensorBoard(log_dir=log_dir, histogram_freq=1)

callbacks_list = [checkpoint, early_stop, reduce_lr, tensorboard_callback]

# --- 3. THE TRAINING RUN ---
print(f"🚀 Starting training for {MODEL_NAME}...")
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks_list,
    verbose=1
)

# --- 4. VISUALIZE RESULTS ---
def plot_training_results(history):
    # Print keys to debug if names don't match
    print("History keys found:", history.history.keys())

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

    # --- 1. Steering Loss ---
    # We use .get() to avoid crashing if the name is slightly different
    ax1.plot(history.history.get('steering_output_loss', []), label='Train Steering')
    ax1.plot(history.history.get('val_steering_output_loss', []), label='Val Steering')
    ax1.set_title('Steering Accuracy (MSE)')
    ax1.set_xlabel('Epochs')
    ax1.set_ylabel('Loss')
    ax1.legend() # Corrected from set_legend()

    # --- 2. Mask Loss ---
    ax2.plot(history.history.get('mask_output_loss', []), label='Train Mask')
    ax2.plot(history.history.get('val_mask_output_loss', []), label='Val Mask')
    ax2.set_title('Road Mask Accuracy (BCE)')
    ax2.set_xlabel('Epochs')
    ax2.set_ylabel('Loss')
    ax2.legend() # Corrected from set_legend()

    plt.tight_layout()
    plt.show()

# Run the corrected plot
plot_training_results(history)

In [ ]:
def debug_dataset_samples(dataset, num_samples=3):
    for images, targets in dataset.take(1):
        img_batch = images.numpy()
        mask_batch = targets['mask_output'].numpy()

        for i in range(num_samples):
            fig, axes = plt.subplots(1, 2, figsize=(10, 5))

            # Show the 'Current' frame of the 9-channel stack (Channels 0-2)
            axes[0].imshow(img_batch[i][:, :, 0:3])
            axes[0].set_title(f"Sample {i}: Input Image (t)")

            # Show the target mask
            # We use vmin/vmax to see if the mask is truly binary (0 and 1)
            axes[1].imshow(mask_batch[i].squeeze(), cmap='gray', vmin=0, vmax=1)
            axes[1].set_title(f"Sample {i}: Target Mask")

            for ax in axes: ax.axis('off')
            plt.show()

# Run the debug check
debug_dataset_samples(train_ds)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def test_model_on_samples(model, dataset, num_samples=5):
    # Take one batch from the validation set
    for images, targets in dataset.take(1):
        # Generate predictions for the whole batch
        # predictions[0] = steering, predictions[1] = mask
        predictions = model.predict(images, verbose=0)

        pred_steerings = predictions[0]
        pred_masks = predictions[1]

        true_steerings = targets['steering_output'].numpy()
        true_masks = targets['mask_output'].numpy()

        print(f"{'Sample':<10} | {'True Steering':<15} | {'Pred Steering':<15} | {'Error'}")
        print("-" * 55)

        for i in range(num_samples):
            true_s = true_steerings[i]
            pred_s = pred_steerings[i][0]
            error = abs(true_s - pred_s)

            print(f"{i:<10} | {true_s:>14.4f} | {pred_s:>14.4f} | {error:.4f}")

            # --- Visualization ---
            fig, axes = plt.subplots(1, 3, figsize=(15, 5))

            # 1. Input Image (Current frame t)
            # Channel slice 0:3 is the most recent frame
            axes[0].imshow(images[i][:, :, 0:3])
            axes[0].set_title(f"Actual Frame (t)\nTrue Steer: {true_s:.2f}")

            # 2. Ground Truth Mask (The perfect IDs we mapped)
            axes[1].imshow(true_masks[i].squeeze(), cmap='gray')
            axes[1].set_title("Target (Ground Truth) Mask")

            # 3. Predicted Mask (What the AI 'thinks' is road)
            axes[2].imshow(pred_masks[i].squeeze(), cmap='jet')
            axes[2].set_title(f"AI Predicted Mask\nPred Steer: {pred_s:.2f}")

            for ax in axes:
                ax.axis('off')
            plt.show()

# Run the test on your validation dataset
test_model_on_samples(model, val_ds)

Quantization for TPU

In [ ]:
import tensorflow as tf

# 1. Prepare Representative Dataset (CRITICAL for Coral)
# We take 100 samples from your existing train_ds to calibrate quantization
def representative_data_gen():
    for input_value, _ in train_ds.take(100):
        # The Coral needs the input to be the exact shape (1, 96, 160, 9)
        # Our batch is (32, 96, 160, 9), so we yield one at a time
        for i in range(input_value.shape[0]):
            yield [input_value[i:i+1]]

# 2. Setup Converter
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = representative_data_gen

# 3. Enforce Full Integer Quantization
# This ensures NO float operations remain, which is a Coral requirement
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type = tf.uint8  # Coral prefers uint8 or int8 inputs
converter.inference_output_type = tf.uint8 # Consistent with input

# 4. Convert and Save
tflite_model_quant = converter.convert()

with open('pro_kart_model_quant.tflite', 'wb') as f:
    f.write(tflite_model_quant)

print("✅ Success! Quantized TFLite model generated.")

Use TPU Edge Compiler

Download model for testing

In [ ]:
from google.colab import files
# Replace with the actual name of your saved file
files.download(MODEL_SAVE_PATH)

Local Inference Test

In [ ]:
import matplotlib.pyplot as plt
import random
import cv2
import numpy as np
import os
import pandas as pd

def verify_model_visual(num_samples=5):
    # 1. Load the Master CSV
    if 'MASTER_CSV_PATH' not in globals():
        print("❌ Error: MASTER_CSV_PATH not defined.")
        return
    df = pd.read_csv(MASTER_CSV_PATH)

    # 2. Pick random samples
    sample_indices = random.sample(range(len(df)), num_samples)

    # Setup the plot
    fig, axes = plt.subplots(2, num_samples, figsize=(20, 8))

    # 3. Dynamic Shape Detection
    # model.input_shape usually returns (None, Height, Width, Channels)
    expected_h = model.input_shape[1]
    expected_w = model.input_shape[2]
    print(f"🔍 Debug: Model expects Height={expected_h}, Width={expected_w}")

    for i, idx in enumerate(sample_indices):
        row = df.iloc[idx]
        img_path = os.path.join(MASTER_IMAGE_DIR, os.path.basename(row['image']))

        # 4. Load Image
        img_raw = cv2.imread(img_path)
        if img_raw is None:
            print(f"⚠️ Could not load: {img_path}")
            continue

        img_rgb = cv2.cvtColor(img_raw, cv2.COLOR_BGR2RGB)

        # 5. THE CRITICAL RESIZE
        # cv2.resize wants (WIDTH, HEIGHT). Our model wants (HEIGHT, WIDTH).
        # We use the model's own expected dimensions to be 100% sure.
        img_resized = cv2.resize(img_rgb, (expected_w, expected_h))

        # 6. Normalize and Batch
        # Shape becomes (1, expected_h, expected_w, 3)
        img_input = np.expand_dims(img_resized.astype(np.float32) / 255.0, axis=0)

        if i == 0:
            print(f"🔍 Debug: Input Tensor Shape = {img_input.shape}")

        # 7. Prediction with Unpacking
        try:
            preds = model.predict(img_input, verbose=0)
            # preds[0] = Steering, preds[1] = Mask
            pred_steering = float(preds[0][0])
            pred_mask = preds[1][0]
        except Exception as e:
            print(f"❌ Prediction failed: {e}")
            continue

        actual_steering = row['steering']

        # 8. Plotting - Camera Row
        ax_img = axes[0, i]
        ax_img.imshow(img_resized)
        error = abs(actual_steering - pred_steering)
        title_color = 'green' if error < 0.15 else 'orange' if error < 0.3 else 'red'
        ax_img.set_title(f"Act: {actual_steering:.2f}\nPred: {pred_steering:.2f}", color=title_color)
        ax_img.axis('off')

        # 9. Plotting - Mask Row
        ax_mask = axes[1, i]
        # Squeeze removes the (1) channel for grayscale plotting
        ax_mask.imshow(pred_mask.squeeze(), cmap='gray')
        ax_mask.set_title("AI Road Mask")
        ax_mask.axis('off')

    plt.tight_layout()
    plt.show()

# Run it
verify_model_visual(num_samples=6)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Load the master list
df = pd.read_csv("/content/drive/MyDrive/KartProject/master_dataset/driving_log.csv")

# Create a Histogram
plt.figure(figsize=(10, 5))
plt.hist(df['steering'], bins=50, color='skyblue', edgecolor='black')
plt.title(f"Master Dataset Balance ({len(df)} total frames)")
plt.xlabel("Steering Angle (Negative = Left, Positive = Right)")
plt.ylabel("Number of Frames")
plt.grid(axis='y', alpha=0.75)
plt.show()